# nf-core/fetchngs Sandwich Wrapper

**Genesis Workbench** — Zero-Fork Nextflow Orchestration

This notebook implements a **sandwich wrapper pattern** that keeps `nf-core/fetchngs` as a completely untouched black box. All Eisai-specific logic lives in the "bread" — Databricks-native pre-flight and post-flight layers — so that upgrading nf-core requires only bumping the version pin.

```
┌─────────────────────────────────────────────────────────────┐
│              PRE-FLIGHT (Databricks-native)                  │
│  ┌───────────┐ ┌──────────────┐ ┌────────────────────────┐  │
│  │ Validate  │ │ Detect repo  │ │ Generate               │  │
│  │ accession │→│ type (SRA/   │→│ eisai_fetchngs.config  │  │
│  │ IDs       │ │ ENA/GEO/Syn) │ │ (process selectors)    │  │
│  └───────────┘ └──────────────┘ └────────────────────────┘  │
│          │                                │                  │
│          ▼                                ▼                  │
│  ┌───────────────────────────────────────────────────┐      │
│  │ Register run in Delta audit table                 │      │
│  └───────────────────────────────────────────────────┘      │
├─────────────────────────────────────────────────────────────┤
│              BLACK BOX (untouched nf-core)                   │
│  ┌───────────────────────────────────────────────────┐      │
│  │  nextflow run nf-core/fetchngs -r <version>       │      │
│  │    -c eisai_fetchngs.config                       │      │
│  │    --input ids.csv                                │      │
│  │    --outdir <volume_path>                         │      │
│  │    --nf_core_pipeline <target>                    │      │
│  │    --download_method <aspera|ftp|sratools>        │      │
│  └───────────────────────────────────────────────────┘      │
├─────────────────────────────────────────────────────────────┤
│              POST-FLIGHT (Databricks-native)                 │
│  ┌──────────┐ ┌───────────┐ ┌─────────────┐ ┌───────────┐  │
│  │ Parse    │ │ Catalog   │ │ FASTQ QC    │ │ Trigger   │  │
│  │ execution│→│ FASTQs &  │→│ gates       │→│ downstream│  │
│  │ trace    │ │ metadata  │ │ (size, N)   │ │ pipeline  │  │
│  │ → Delta  │ │ → Delta   │ │             │ │ (auto)    │  │
│  └──────────┘ └───────────┘ └─────────────┘ └───────────┘  │
└─────────────────────────────────────────────────────────────┘
```

**Supported Accession Types**:
| Type | Pattern | Repository | Example |
|------|---------|-----------|----------|
| Run | SRR/ERR/DRR | SRA/ENA/DDBJ | SRR12345678 |
| Study | SRP/ERP/DRP | SRA/ENA/DDBJ | SRP123456 |
| BioProject | PRJNA/PRJEB/PRJDB | NCBI/EBI/DDBJ | PRJNA123456 |
| GEO Series | GSE | NCBI GEO | GSE204716 |
| Synapse | syn | Synapse | syn1234567 |

**Downstream auto-trigger**: When `trigger_downstream=true` and QC passes, this wrapper submits the appropriate nf-core sandwich wrapper (scrnaseq, rnaseq) as a Databricks job, passing the fetchngs-generated samplesheet directly.

In [0]:
# Widget parameters — set by Databricks Jobs API from Streamlit form
dbutils.widgets.text("accession_ids", "", "Accession IDs (comma or newline separated)")
dbutils.widgets.text("output_dir", "", "Output Directory (UC Volume path)")
dbutils.widgets.text("pipeline_version", "1.12.0", "nf-core/fetchngs Version")
dbutils.widgets.dropdown("nf_core_pipeline", "scrnaseq", ["scrnaseq", "rnaseq", "taxprofiler", "atacseq", "none"], "Target Downstream Pipeline")
dbutils.widgets.dropdown("download_method", "sratools", ["sratools", "aspera", "ftp"], "Download Method")
dbutils.widgets.dropdown("genome", "GRCh38", ["GRCh38", "GRCm39", "GRCm38"], "Reference Genome")
dbutils.widgets.text("extra_args", "", "Extra Nextflow Args")
dbutils.widgets.dropdown("qc_gate_enabled", "true", ["true", "false"], "Enable FASTQ QC Gates")
dbutils.widgets.text("min_fastq_size_mb", "10", "Min FASTQ Size (MB) QC Gate")
dbutils.widgets.dropdown("trigger_downstream", "false", ["true", "false"], "Auto-trigger Downstream Pipeline")
dbutils.widgets.text("project_name", "", "Project Name (for audit trail)")
dbutils.widgets.text("catalog", "dhbl_discovery_us_dev", "Unity Catalog")
dbutils.widgets.text("schema", "genesis_schema", "Schema")

# Resolve all widgets
accession_ids_raw = dbutils.widgets.get("accession_ids")
output_dir = dbutils.widgets.get("output_dir")
pipeline_version = dbutils.widgets.get("pipeline_version")
nf_core_pipeline = dbutils.widgets.get("nf_core_pipeline")
download_method = dbutils.widgets.get("download_method")
genome = dbutils.widgets.get("genome")
extra_args = dbutils.widgets.get("extra_args") or None
qc_gate_enabled = dbutils.widgets.get("qc_gate_enabled") == "true"
min_fastq_size_mb = float(dbutils.widgets.get("min_fastq_size_mb"))
trigger_downstream = dbutils.widgets.get("trigger_downstream") == "true"
project_name = dbutils.widgets.get("project_name") or "unnamed"
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

import time
run_id = f"fetchngs_{project_name}_{int(time.time())}"

print(f"\n{'='*60}")
print(f"  Sandwich Wrapper Run: {run_id}")
print(f"{'='*60}")
print(f"  Output dir:     {output_dir}")
print(f"  Pipeline:       nf-core/fetchngs v{pipeline_version}")
print(f"  Downstream:     nf-core/{nf_core_pipeline}")
print(f"  Download:       {download_method}")
print(f"  Genome:         {genome}")
print(f"  QC gates:       {'ON' if qc_gate_enabled else 'OFF'}")
print(f"  Auto-trigger:   {'ON' if trigger_downstream else 'OFF'}")
print(f"  Catalog:        {catalog}.{schema}")
print(f"{'='*60}")

---
## 🔵 Layer 1 — Pre-Flight (Databricks-Native)

Everything before `nextflow run`. Validates accession IDs, detects repository types, generates the institutional config overlay, and registers the run in a Delta audit table. **Zero nf-core modifications.**

In [0]:
import os, sys, re, subprocess, json
from datetime import datetime, timezone

# ── Validate Nextflow is installed ──
try:
    nxf_out = subprocess.check_output(["nextflow", "-version"], stderr=subprocess.STDOUT, text=True)
    nxf_version = [l for l in nxf_out.strip().split("\n") if "version" in l.lower()]
    print(f"\u2705 Nextflow: {nxf_version[0].strip() if nxf_version else nxf_out.strip().split(chr(10))[0]}")
except FileNotFoundError:
    raise RuntimeError(
        "\u274c Nextflow not found. Cluster needs init script: "
        "/Volumes/dhbl_discovery_us_dev/genesis_schema/scanpy_reference/init_scripts/install_nextflow.sh"
    )

# ── Parse accession IDs ──
assert accession_ids_raw.strip(), "\u274c No accession IDs provided. Supply comma or newline-separated IDs."

# Split on commas, newlines, semicolons, or whitespace
accession_ids = [x.strip() for x in re.split(r'[,;\n\s]+', accession_ids_raw.strip()) if x.strip()]

# ── Validate each accession against known patterns ──
ACCESSION_PATTERNS = {
    "SRA Run": re.compile(r'^[SED]RR\d{6,}$'),
    "SRA Study": re.compile(r'^[SED]RP\d{6,}$'),
    "BioProject": re.compile(r'^PRJ[A-Z]{2}\d+$'),
    "GEO Series": re.compile(r'^GSE\d+$'),
    "Synapse": re.compile(r'^syn\d+$'),
}

validated_ids = []
id_types = {}

for acc_id in accession_ids:
    matched = False
    for acc_type, pattern in ACCESSION_PATTERNS.items():
        if pattern.match(acc_id):
            validated_ids.append(acc_id)
            id_types[acc_id] = acc_type
            matched = True
            break
    if not matched:
        print(f"\u26a0\ufe0f  Unrecognized accession format: {acc_id} (will pass through — fetchngs may still handle it)")
        validated_ids.append(acc_id)
        id_types[acc_id] = "Unknown"

assert validated_ids, "\u274c No valid accession IDs found after parsing."

# ── Ensure output directory exists ──
assert output_dir, "\u274c Output directory must be specified"
os.makedirs(output_dir, exist_ok=True)

# ── Write input IDs file (one per line, as required by fetchngs) ──
ids_file = os.path.join(output_dir, "accession_ids.csv")
with open(ids_file, "w") as f:
    for acc_id in validated_ids:
        f.write(f"{acc_id}\n")

# ── Summary ──
type_counts = {}
for t in id_types.values():
    type_counts[t] = type_counts.get(t, 0) + 1

print(f"\u2705 Accession IDs: {len(validated_ids)} validated")
print(f"   Input file: {ids_file}")
print(f"   Types:")
for acc_type, count in sorted(type_counts.items()):
    print(f"     {acc_type}: {count}")
print(f"   First 5: {validated_ids[:5]}")
print(f"\u2705 Output dir: {output_dir}")

In [0]:
def generate_eisai_fetchngs_config(output_dir: str, download_method: str) -> str:
    """
    Generate eisai_fetchngs.config — the institutional Nextflow config overlay.

    This is the KEY to zero-fork customization. Process selectors let us
    override resources, containers, publishDir, and behavior per-process
    WITHOUT modifying any nf-core pipeline code.

    When nf-core/fetchngs is upgraded:
    - If a process name changes → update the withName selector here
    - If a new process is added → optionally add a selector
    - If a process is removed → the selector is harmlessly ignored

    This config is passed via: nextflow run ... -c eisai_fetchngs.config
    """
    config = f"""\
// =================================================================
// eisai_fetchngs.config — Institutional overlay for nf-core/fetchngs
// Generated: {datetime.now(timezone.utc).isoformat()}
// Pattern:   Sandwich Wrapper (zero-fork)
//
// This file is GENERATED per-run. Edit the generate_eisai_fetchngs_config()
// function in the sandwich wrapper notebook, NOT this file.
// =================================================================

// ---- Global resource defaults ----
process {{
    cpus   = {{ Math.min(params.max_cpus ?: 4, Runtime.runtime.availableProcessors()) }}
    memory = {{ Math.min((params.max_memory ?: '16.GB') as nextflow.util.MemoryUnit,
                        new nextflow.util.MemoryUnit(Runtime.runtime.maxMemory())) }}
    time   = '4.h'

    // Retry strategy: bump resources on transient network failures
    errorStrategy = {{ task.exitStatus in [104, 134, 137, 139, 143, 247] ? 'retry' : 'finish' }}
    maxRetries    = 3

    // ---- Process-specific overrides ----
    withName: 'SRA_IDS_TO_RUNINFO' {{
        cpus   = 1
        memory = '2.GB'
        time   = '1.h'
        errorStrategy = {{ task.exitStatus in [1, 104, 134, 137, 139, 143, 247] ? 'retry' : 'finish' }}
        maxRetries = 5
    }}

    withName: 'SRA_RUNINFO_TO_FTP' {{
        cpus   = 1
        memory = '2.GB'
        time   = '30.m'
    }}

    withName: 'ASPERA_CLI' {{
        cpus   = 2
        memory = '4.GB'
        time   = '6.h'
        errorStrategy = {{ task.exitStatus in [1, 104, 134, 137, 139, 143, 247] ? 'retry' : 'finish' }}
        maxRetries = 3
    }}

    withName: 'SRATOOLS_FASTERQDUMP' {{
        cpus   = {{ Math.min(4, Runtime.runtime.availableProcessors()) }}
        memory = {{ task.attempt == 1 ? '8.GB' : '16.GB' }}
        time   = '8.h'
        errorStrategy = {{ task.exitStatus in [1, 104, 134, 137, 139, 143, 247] ? 'retry' : 'finish' }}
        maxRetries = 3
    }}

    withName: 'SRATOOLS_PREFETCH' {{
        cpus   = 2
        memory = '8.GB'
        time   = '4.h'
        errorStrategy = {{ task.exitStatus in [1, 104, 134, 137, 139, 143, 247] ? 'retry' : 'finish' }}
        maxRetries = 3
    }}

    withName: 'FTP_DOWNLOAD|WGET_DOWNLOAD' {{
        cpus   = 1
        memory = '2.GB'
        time   = '6.h'
        errorStrategy = {{ task.exitStatus in [1, 104, 134, 137, 139, 143, 247] ? 'retry' : 'finish' }}
        maxRetries = 3
    }}

    withName: 'SYNAPSE_GET' {{
        cpus   = 1
        memory = '4.GB'
        time   = '4.h'
    }}

    withName: 'MD5_CHECK|FASTQ_CHECKSUM' {{
        cpus   = 1
        memory = '1.GB'
        time   = '30.m'
    }}

    withName: 'MULTIQC|CUSTOM_DUMPSOFTWAREVERSIONS' {{
        cpus   = 1
        memory = '2.GB'
    }}
}}

trace {{
    enabled   = true
    file      = '{output_dir}/trace/execution_trace.txt'
    sep       = '\\t'
    fields    = 'task_id,hash,native_id,name,status,exit,submit,start,complete,duration,realtime,%cpu,%mem,rss,vmem,peak_rss,peak_vmem,rchar,wchar,read_bytes,write_bytes'
    overwrite = true
}}

timeline {{
    enabled   = true
    file      = '{output_dir}/trace/timeline.html'
    overwrite = true
}}

report {{
    enabled   = true
    file      = '{output_dir}/trace/report.html'
    overwrite = true
}}

dag {{
    enabled   = true
    file      = '{output_dir}/trace/dag.html'
    overwrite = true
}}

// Use MAMBA (not conda) — mamba supports --mkdir flag that newer conda dropped.
// Each process gets its own mamba env with correct dependencies
// (sra-tools for SRATOOLS processes, python for SRA_IDS_TO_RUNINFO, etc.)
conda {{
    enabled        = true
    useMamba       = true
    cacheDir       = '/tmp/nf-conda-cache'
    createTimeout  = '1.h'
}}

cleanup = true
"""
    config_path = os.path.join(output_dir, "eisai_fetchngs.config")
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "trace"), exist_ok=True)

    with open(config_path, "w") as f:
        f.write(config)

    return config_path


eisai_config_path = generate_eisai_fetchngs_config(output_dir, download_method)
print(f"\u2705 Institutional config: {eisai_config_path}")
print(f"   Trace output:        {output_dir}/trace/")
print(f"   Download method:     {download_method}")
print(f"   Conda:              ENABLED via Mamba (supports --mkdir)")
print(f"\nConfig preview (first 30 lines):")
with open(eisai_config_path) as f:
    for i, line in enumerate(f):
        if i >= 30:
            print("   ...")
            break
        print(f"   {line}", end="")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, DoubleType, BooleanType,
)

AUDIT_TABLE = f"{catalog}.{schema}.nextflow_run_audit"

run_started_at = datetime.now(timezone.utc)

# Schema matches the shared audit table used by all sandwich wrappers
# fetchngs-specific: download_method, nf_core_pipeline, n_accessions
audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("project_name", StringType(), True),
    StructField("pipeline", StringType(), True),
    StructField("pipeline_version", StringType(), True),
    StructField("aligner", StringType(), True),
    StructField("genome", StringType(), True),
    StructField("chemistry", StringType(), True),
    StructField("fastq_dir", StringType(), True),
    StructField("output_dir", StringType(), True),
    StructField("n_fastq_files", IntegerType(), True),
    StructField("fastq_size_gb", DoubleType(), True),
    StructField("n_samples", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("completed_at", TimestampType(), True),
    StructField("elapsed_minutes", DoubleType(), True),
    StructField("n_cells", IntegerType(), True),
    StructField("median_genes_per_cell", IntegerType(), True),
    StructField("qc_gate_passed", BooleanType(), True),
    StructField("scanpy_triggered", BooleanType(), True),
    StructField("error_message", StringType(), True),
    StructField("wrapper_version", StringType(), True),
])

# Insert initial "running" record
# Re-use shared schema columns; fetchngs-specific data stored in params JSON
initial_row = Row(
    run_id=run_id,
    project_name=project_name,
    pipeline="nf-core/fetchngs",
    pipeline_version=pipeline_version,
    aligner=download_method,  # re-use aligner col for download method
    genome=genome,
    chemistry=nf_core_pipeline,  # re-use chemistry col for target pipeline
    fastq_dir=ids_file,  # input file path
    output_dir=output_dir,
    n_fastq_files=len(validated_ids),  # re-use for n_accessions
    fastq_size_gb=None,
    n_samples=len(validated_ids),
    status="running",
    started_at=run_started_at,
    completed_at=None,
    elapsed_minutes=None,
    n_cells=None,
    median_genes_per_cell=None,
    qc_gate_passed=None,
    scanpy_triggered=None,
    error_message=None,
    wrapper_version="1.0.0",
)

try:
    df = spark.createDataFrame([initial_row], schema=audit_schema)
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(AUDIT_TABLE)
    print(f"\u2705 Run registered: {AUDIT_TABLE}")
    print(f"   run_id: {run_id}")
    print(f"   status: running")
    print(f"   accessions: {len(validated_ids)}")
except Exception as e:
    # Non-fatal: audit failure shouldn't block the pipeline
    print(f"\u26a0\ufe0f  Audit table write failed (non-fatal): {e}")
    print(f"   Pipeline will continue without audit trail.")

print(f"\n{'='*60}")
print(f"  Pre-flight complete. Handing off to nf-core/fetchngs.")
print(f"{'='*60}")

---
## 🟢 Layer 2 — nf-core/fetchngs (Black Box)

The pipeline runs as an **opaque subprocess**. The only interface points are:
- `-c eisai_fetchngs.config` (institutional overlay from Layer 1)
- `--input`, `--outdir`, `--nf_core_pipeline`, `--download_method` (standard nf-core params)

**ZERO modifications** to the pipeline code. Upgrading = bumping the `-r` version tag.

In [0]:
import time as _time

# fetchngs v1.12.0 only accepts these values for --nf_core_pipeline
# scrnaseq is NOT a valid fetchngs choice (it has a different samplesheet format)
# When target is scrnaseq or none, we omit the flag and use the default samplesheet
FETCHNGS_VALID_PIPELINES = {"rnaseq", "atacseq", "viralrecon", "taxprofiler"}

# Build the nf-core/fetchngs command
cmd = [
    "nextflow", "run", f"nf-core/fetchngs",
    "-r", pipeline_version,
    "-c", eisai_config_path,
    "--input", ids_file,
    "--outdir", output_dir,
    "--download_method", download_method,
]

# Only pass --nf_core_pipeline if it's a valid fetchngs choice
if nf_core_pipeline in FETCHNGS_VALID_PIPELINES:
    cmd.extend(["--nf_core_pipeline", nf_core_pipeline])
else:
    print(f"   Note: '{nf_core_pipeline}' not in fetchngs valid choices {FETCHNGS_VALID_PIPELINES}")
    print(f"   Omitting --nf_core_pipeline (will use default samplesheet format)")

# Append extra args if provided
if extra_args:
    cmd.extend(extra_args.split())

print(f"\U0001f680 Running nf-core/fetchngs v{pipeline_version}")
print(f"   Download method: {download_method}")
print(f"   Target pipeline: {nf_core_pipeline}")
print(f"   Accessions: {len(validated_ids)}")
print(f"   Command:")
print(f"   {' '.join(cmd)}")
print(f"   Started: {_time.strftime('%Y-%m-%d %H:%M:%S UTC', _time.gmtime())}")
print(f"\n{'='*60}\n")

# ── Use LOCAL disk for Nextflow working directory ──
# UC Volumes (FUSE) don't support file locking / POSIX ops needed by
# Nextflow's plugin resolver and work directory. Use local SSD/EBS instead.
nxf_work_dir = "/local_disk0/nextflow_work"
os.makedirs(nxf_work_dir, exist_ok=True)

# ── Create conda/mamba wrapper to strip --mkdir flag ──
# Nextflow 24.10.4 passes --mkdir to conda/mamba, but newer Miniforge3
# dropped support for it (--prefix auto-creates dirs now).
# Wrapper intercepts and strips it before calling the real binary.
# Also sets CONDA_ALWAYS_YES=true to suppress interactive prompts
# (mamba 2.x ignores --yes in some contexts and prompts for [Y/n]).
wrapper_dir = "/local_disk0/nxf_bin_wrappers"
os.makedirs(wrapper_dir, exist_ok=True)

for tool in ["conda", "mamba"]:
    # Find the real binary (init script installs to /opt/miniforge3)
    real_bin = f"/opt/miniforge3/bin/{tool}"
    wrapper_path = os.path.join(wrapper_dir, tool)
    wrapper_content = f"""#!/bin/bash
# Wrapper: strips --mkdir and forces auto-confirm for {tool}
export CONDA_ALWAYS_YES=true
args=()
for arg in "$@"; do
    [[ "$arg" != "--mkdir" ]] && args+=("$arg")
done
# Only append --yes for create/install (not activate/run/other subcommands)
if [[ " ${{args[*]}} " == *" create "* ]] || [[ " ${{args[*]}} " == *" install "* ]]; then
    exec {real_bin} "${{args[@]}}" --yes
else
    exec {real_bin} "${{args[@]}}"
fi
"""
    with open(wrapper_path, "w") as f:
        f.write(wrapper_content)
    os.chmod(wrapper_path, 0o755)

print(f"   \u2705 conda/mamba wrappers: {wrapper_dir} (strips --mkdir, auto-confirms)")

# Set NXF_HOME to local disk so .nextflow/ (plugins, cache) lives on real FS
nxf_env = os.environ.copy()
nxf_env["NXF_HOME"] = "/local_disk0/.nextflow"
nxf_env["NXF_WORK"] = nxf_work_dir
nxf_env["CONDA_ALWAYS_YES"] = "true"  # Belt-and-suspenders: also set in parent env
# Prepend wrapper dir to PATH so Nextflow finds our patched conda/mamba first
nxf_env["PATH"] = f"{wrapper_dir}:{nxf_env['PATH']}"

print(f"   NXF_HOME: {nxf_env['NXF_HOME']}")
print(f"   NXF_WORK: {nxf_work_dir}")
print(f"   outdir (UC Volume): {output_dir}")

# Run with live output streaming
log_path = os.path.join(output_dir, "nextflow_fetchngs.log")
start_time = _time.time()

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=nxf_work_dir,  # Run from local disk, NOT UC Volume
    env=nxf_env,
)

with open(log_path, "w") as log_file:
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

process.wait()
nf_elapsed = _time.time() - start_time
nf_exit_code = process.returncode

print(f"\n{'='*60}")
if nf_exit_code == 0:
    print(f"\u2705 fetchngs completed in {nf_elapsed/60:.1f} minutes")
else:
    print(f"\u274c fetchngs FAILED in {nf_elapsed/60:.1f} minutes (exit code: {nf_exit_code})")
    print(f"   Log: {log_path}")
    # Don't raise yet — let post-flight capture the failure in audit table

---
## 🟡 Layer 3 — Post-Flight (Databricks-Native)

Everything after `nextflow run` completes. Parses execution traces into Delta, catalogs downloaded FASTQs and metadata, runs QC gates on download quality, and optionally triggers the downstream pipeline. **Zero nf-core modifications.**

In [0]:
import pandas as pd

TRACE_TABLE = f"{catalog}.{schema}.nextflow_execution_traces"
trace_path = os.path.join(output_dir, "trace", "execution_trace.txt")

trace_df = None
if os.path.exists(trace_path):
    try:
        trace_pdf = pd.read_csv(trace_path, sep="\t")
        trace_pdf["run_id"] = run_id
        trace_pdf["project_name"] = project_name
        trace_pdf["pipeline"] = "fetchngs"
        trace_pdf["pipeline_version"] = pipeline_version
        trace_pdf["ingested_at"] = datetime.now(timezone.utc)

        trace_df = spark.createDataFrame(trace_pdf)
        trace_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TRACE_TABLE)

        print(f"\u2705 Execution trace: {len(trace_pdf)} tasks \u2192 {TRACE_TABLE}")
        print(f"\nProcess summary:")
        summary = trace_pdf.groupby("name").agg(
            count=("task_id", "count"),
            status_counts=("status", lambda x: dict(x.value_counts())),
        )
        for _, row in summary.iterrows():
            print(f"   {row.name}: {row['count']} tasks, status={row['status_counts']}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Trace ingestion failed (non-fatal): {e}")
else:
    print(f"\u26a0\ufe0f  No execution trace found at {trace_path}")
    if nf_exit_code != 0:
        print(f"   (Expected \u2014 pipeline failed before producing trace)")

In [0]:
import glob as _glob

SAMPLES_TABLE = f"{catalog}.{schema}.fetchngs_samples"
METADATA_TABLE = f"{catalog}.{schema}.fetchngs_metadata"

# ── Locate the auto-generated samplesheet ──
# fetchngs outputs samplesheet at: <outdir>/samplesheet/samplesheet.csv
samplesheet_candidates = [
    os.path.join(output_dir, "samplesheet", "samplesheet.csv"),
    os.path.join(output_dir, "samplesheet", "nf-core-fetchngs_samplesheet.csv"),
] + _glob.glob(os.path.join(output_dir, "samplesheet", "*.csv"))

samplesheet_path = None
for candidate in samplesheet_candidates:
    if os.path.exists(candidate):
        samplesheet_path = candidate
        break

n_samples_downloaded = 0
fastq_catalog = []

if samplesheet_path:
    print(f"\u2705 Samplesheet found: {samplesheet_path}")
    samplesheet_df = pd.read_csv(samplesheet_path)
    print(f"   Columns: {list(samplesheet_df.columns)}")
    print(f"   Samples: {len(samplesheet_df)}")
    print(f"\n   Preview (first 5):")
    print(samplesheet_df.head().to_string(index=False))

    # Catalog each sample's FASTQ files
    for _, row in samplesheet_df.iterrows():
        sample_id = row.get("sample", row.get("id", "unknown"))
        fastq_1 = row.get("fastq_1", "")
        fastq_2 = row.get("fastq_2", "")
        single_end = pd.isna(fastq_2) or str(fastq_2).strip() == ""
        strandedness = row.get("strandedness", "auto")

        # Compute file sizes
        size_1_mb = os.path.getsize(fastq_1) / (1024**2) if fastq_1 and os.path.exists(str(fastq_1)) else 0
        size_2_mb = os.path.getsize(fastq_2) / (1024**2) if fastq_2 and not single_end and os.path.exists(str(fastq_2)) else 0
        total_size_mb = size_1_mb + size_2_mb

        fastq_catalog.append({
            "run_id": run_id,
            "project_name": project_name,
            "sample_id": str(sample_id),
            "fastq_1": str(fastq_1) if fastq_1 else None,
            "fastq_2": str(fastq_2) if not single_end else None,
            "single_end": single_end,
            "strandedness": str(strandedness) if strandedness else "auto",
            "download_method": download_method,
            "source_accession": str(sample_id),
            "downloaded_at": datetime.now(timezone.utc),
            "size_mb": round(total_size_mb, 2),
            "pipeline_target": nf_core_pipeline,
        })

    n_samples_downloaded = len(fastq_catalog)

    # Write FASTQ catalog to Delta
    if fastq_catalog:
        try:
            samples_spark_df = spark.createDataFrame(pd.DataFrame(fastq_catalog))
            samples_spark_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(SAMPLES_TABLE)
            print(f"\n\u2705 FASTQ catalog: {n_samples_downloaded} samples \u2192 {SAMPLES_TABLE}")
        except Exception as e:
            print(f"\u26a0\ufe0f  FASTQ catalog write failed (non-fatal): {e}")

    # ── Also ingest raw SRA metadata if available ──
    sra_metadata_path = os.path.join(output_dir, "custom", "sra_metadata.tsv")
    if not os.path.exists(sra_metadata_path):
        # Try alternate locations
        sra_metadata_path = os.path.join(output_dir, "metadata", "sra_metadata.tsv")

    if os.path.exists(sra_metadata_path):
        try:
            meta_pdf = pd.read_csv(sra_metadata_path, sep="\t")
            meta_pdf["run_id"] = run_id
            meta_pdf["project_name"] = project_name
            meta_pdf["ingested_at"] = datetime.now(timezone.utc)

            meta_spark_df = spark.createDataFrame(meta_pdf)
            meta_spark_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(METADATA_TABLE)
            print(f"\u2705 SRA metadata: {len(meta_pdf)} rows \u2192 {METADATA_TABLE}")
        except Exception as e:
            print(f"\u26a0\ufe0f  SRA metadata write failed (non-fatal): {e}")
    else:
        print(f"\u26a0\ufe0f  No SRA metadata TSV found (normal for GEO/Synapse accessions)")
else:
    if nf_exit_code == 0:
        print(f"\u26a0\ufe0f  No samplesheet found in {output_dir}/samplesheet/")
        print(f"   Searched: {samplesheet_candidates[:3]}")
    else:
        print(f"\u26a0\ufe0f  No samplesheet (expected \u2014 pipeline failed)")

In [0]:
qc_passed = True
qc_reasons = []

if nf_exit_code != 0:
    qc_passed = False
    qc_reasons.append(f"Pipeline failed with exit code {nf_exit_code}")

elif qc_gate_enabled:
    # Gate 1: At least one sample was successfully downloaded
    if n_samples_downloaded == 0:
        qc_passed = False
        qc_reasons.append("No samples downloaded successfully")

    # Gate 2: Samplesheet was generated
    if not samplesheet_path:
        qc_passed = False
        qc_reasons.append("No samplesheet generated by fetchngs")

    # Gate 3: Each FASTQ meets minimum file size threshold
    undersized_samples = []
    for sample in fastq_catalog:
        if sample["size_mb"] < min_fastq_size_mb:
            undersized_samples.append(
                f"{sample['sample_id']} ({sample['size_mb']:.1f} MB < {min_fastq_size_mb} MB)"
            )

    if undersized_samples:
        qc_passed = False
        qc_reasons.append(
            f"{len(undersized_samples)} sample(s) below min size ({min_fastq_size_mb} MB): "
            + "; ".join(undersized_samples[:5])
        )

    # Gate 4: Check FASTQ files actually exist on disk
    missing_files = []
    for sample in fastq_catalog:
        if sample["fastq_1"] and not os.path.exists(sample["fastq_1"]):
            missing_files.append(sample["fastq_1"])
        if sample["fastq_2"] and not os.path.exists(sample["fastq_2"]):
            missing_files.append(sample["fastq_2"])

    if missing_files:
        qc_passed = False
        qc_reasons.append(
            f"{len(missing_files)} FASTQ file(s) missing from disk: "
            + "; ".join(missing_files[:3])
        )

# Report results
if qc_passed:
    print(f"\u2705 QC Gates: ALL PASSED")
    print(f"   Samples downloaded: {n_samples_downloaded}")
    total_size_gb = sum(s["size_mb"] for s in fastq_catalog) / 1024
    print(f"   Total FASTQ size:   {total_size_gb:.2f} GB")
    print(f"   Min file size gate: {min_fastq_size_mb} MB \u2705")
else:
    print(f"\u274c QC Gates: FAILED")
    for reason in qc_reasons:
        print(f"   \u2022 {reason}")

In [0]:
run_completed_at = datetime.now(timezone.utc)
run_elapsed = (run_completed_at - run_started_at).total_seconds() / 60

final_status = (
    "succeeded" if nf_exit_code == 0 and qc_passed
    else "qc_failed" if nf_exit_code == 0 and not qc_passed
    else "failed"
)

# Update audit table with final status
try:
    spark.sql(f"""
        MERGE INTO {AUDIT_TABLE} AS t
        USING (SELECT '{run_id}' AS run_id) AS s
        ON t.run_id = s.run_id
        WHEN MATCHED THEN UPDATE SET
            t.status = '{final_status}',
            t.completed_at = timestamp '{run_completed_at.isoformat()}',
            t.elapsed_minutes = {run_elapsed:.1f},
            t.n_samples = {n_samples_downloaded or 'NULL'},
            t.qc_gate_passed = {str(qc_passed).lower()},
            t.scanpy_triggered = false,
            t.error_message = {f"'{'; '.join(qc_reasons)[:500]}'" if qc_reasons else 'NULL'}
    """)
    print(f"\u2705 Audit table updated: status={final_status}")
except Exception as e:
    print(f"\u26a0\ufe0f  Audit update failed (non-fatal): {e}")

# Trigger downstream pipeline if enabled and QC passed
downstream_triggered = False
downstream_run_id = None

if trigger_downstream and qc_passed and samplesheet_path and nf_core_pipeline != "none":
    try:
        from databricks.sdk import WorkspaceClient

        w = WorkspaceClient()

        # Map nf_core_pipeline to the appropriate sandwich wrapper notebook
        WRAPPER_NOTEBOOKS = {
            "scrnaseq": (
                "/Users/andrew_forman@eisai.com/genesis-workbench"
                "/modules/single_cell/scanpy/scanpy_v0.0.1/notebooks/nextflow_sandwich_wrapper"
            ),
            "rnaseq": (
                "/Users/andrew_forman@eisai.com/genesis-workbench"
                "/modules/bulk_rnaseq/notebooks/nextflow_sandwich_wrapper_rnaseq"
            ),
        }

        wrapper_notebook = WRAPPER_NOTEBOOKS.get(nf_core_pipeline)

        if wrapper_notebook:
            # Determine FASTQ directory from fetchngs output
            fastq_output_dir = os.path.join(output_dir, "fastq")
            if not os.path.isdir(fastq_output_dir):
                # Fallback: some versions put FASTQs directly in outdir
                fastq_output_dir = output_dir

            # Build parameters for the downstream wrapper
            downstream_params = {
                "fastq_dir": fastq_output_dir,
                "output_dir": os.path.join(output_dir, f"{nf_core_pipeline}_results"),
                "genome": genome,
                "project_name": f"{project_name}_from_fetchngs",
                "pipeline_version": "2.7.1" if nf_core_pipeline == "scrnaseq" else "3.16.1",
            }

            downstream_run = w.jobs.submit(
                run_name=f"{nf_core_pipeline} auto-trigger from fetchngs: {run_id}",
                tasks=[{
                    "task_key": f"{nf_core_pipeline}_alignment",
                    "notebook_task": {
                        "notebook_path": wrapper_notebook,
                        "base_parameters": downstream_params,
                        "source": "WORKSPACE",
                    },
                    "new_cluster": {
                        "spark_version": "16.4.x-scala2.12",
                        "node_type_id": "r5.4xlarge",
                        "num_workers": 0,
                        "spark_conf": {
                            "spark.master": "local[*]",
                            "spark.databricks.cluster.profile": "singleNode",
                        },
                        "data_security_mode": "SINGLE_USER",
                    },
                }],
            )

            downstream_triggered = True
            downstream_run_id = downstream_run.run_id
            print(f"\n\U0001f680 Downstream pipeline auto-triggered!")
            print(f"   Pipeline:     nf-core/{nf_core_pipeline}")
            print(f"   Wrapper:      {wrapper_notebook}")
            print(f"   Run ID:       {downstream_run_id}")
            print(f"   FASTQ dir:    {fastq_output_dir}")
            print(f"   Samplesheet:  {samplesheet_path}")

            # Update audit table with trigger status
            try:
                spark.sql(f"""
                    MERGE INTO {AUDIT_TABLE} AS t
                    USING (SELECT '{run_id}' AS run_id) AS s
                    ON t.run_id = s.run_id
                    WHEN MATCHED THEN UPDATE SET t.scanpy_triggered = true
                """)
            except Exception:
                pass

        else:
            print(f"\n\u2139\ufe0f  No sandwich wrapper configured for '{nf_core_pipeline}'")
            print(f"   Samplesheet available at: {samplesheet_path}")
            print(f"   Use manually with: nextflow run nf-core/{nf_core_pipeline} --input {samplesheet_path}")

    except Exception as e:
        print(f"\u26a0\ufe0f  Downstream trigger failed (non-fatal): {e}")

elif trigger_downstream and not qc_passed:
    print(f"\n\u23ed\ufe0f  Downstream NOT triggered \u2014 QC gates failed")
elif trigger_downstream and nf_core_pipeline == "none":
    print(f"\n\u23ed\ufe0f  Downstream NOT triggered \u2014 no target pipeline specified")
elif not trigger_downstream:
    print(f"\n\u23ed\ufe0f  Downstream auto-trigger disabled")

In [0]:
# =============================================================================
# Lineage Logging — Metadata Tier Integration
# Records assets + edges in the genesis_schema metadata tier tables so that
# downstream consumers (GWAS, scanpy, sarek) can trace provenance back to
# the original SRA/GEO accessions.
# =============================================================================

try:
    from genesis_workbench.lineage import LineageLogger

    lineage = LineageLogger(
        module="fetchngs",
        run_id=run_id,
        user_email=triggered_by,
        run_source="nextflow",
        catalog=catalog,
        schema=schema,
    )

    # ── Register input asset: the accession IDs ──
    accession_asset = lineage.register_asset(
        path=os.path.join(output_dir, "ids.csv"),
        asset_type="volume_file",
        tier="bronze",
        format="csv",
        display_name=f"Accession IDs ({project_name})",
        description=f"{n_accessions} accessions: {accession_ids[:100]}...",
        tags={
            "accession_type": accession_type if 'accession_type' in dir() else "mixed",
            "n_accessions": str(n_accessions),
            "download_method": download_method,
            "project_name": project_name,
        },
    )

    # ── Register output assets: downloaded FASTQs (bronze) ──
    fastq_output_dir = os.path.join(output_dir, "fastq")
    if not os.path.isdir(fastq_output_dir):
        fastq_output_dir = output_dir

    fastq_asset = lineage.register_asset(
        path=fastq_output_dir,
        asset_type="volume_file",
        tier="bronze",
        format="fastq.gz",
        display_name=f"FASTQs ({project_name})",
        description=f"{n_samples_downloaded or 0} samples downloaded via nf-core/fetchngs {pipeline_version}",
        tags={
            "genome": genome,
            "n_samples": str(n_samples_downloaded or 0),
            "pipeline": "nf-core/fetchngs",
            "pipeline_version": pipeline_version,
        },
        file_size_bytes=int(total_fastq_size_gb * 1e9) if 'total_fastq_size_gb' in dir() and total_fastq_size_gb else None,
    )

    # ── Register output asset: generated samplesheet (gold — analysis-ready) ──
    samplesheet_asset = None
    if samplesheet_path and os.path.exists(samplesheet_path):
        samplesheet_asset = lineage.register_asset(
            path=samplesheet_path,
            asset_type="volume_file",
            tier="gold",
            format="csv",
            display_name=f"Samplesheet ({project_name})",
            description=f"Auto-generated samplesheet for nf-core/{nf_core_pipeline}",
            tags={
                "target_pipeline": nf_core_pipeline,
                "n_samples": str(n_samples_downloaded or 0),
            },
        )

    # ── Record lineage edges ──
    # Accession IDs → FASTQs (the core data acquisition step)
    lineage.link(
        source=accession_asset,
        target=fastq_asset,
        relationship="consumed_by",
        step="fetchngs_download",
    )

    # FASTQs → Samplesheet (derived output)
    if samplesheet_asset:
        lineage.link(
            source=fastq_asset,
            target=samplesheet_asset,
            relationship="derived_from",
            step="samplesheet_generation",
        )

    # If downstream was triggered, record the trigger edge
    if downstream_triggered and samplesheet_asset:
        # The downstream wrapper will register its own output assets;
        # here we just record that this samplesheet triggered the next pipeline.
        # Use a placeholder target — downstream wrapper will fill in real outputs.
        lineage.link(
            source=samplesheet_asset,
            target=fastq_asset,  # self-reference signals "triggered downstream"
            relationship="triggered_by",
            step=f"auto_trigger_{nf_core_pipeline}",
        )

    # ── Register the run in run_catalog ──
    lineage.register_run(
        run_name=f"fetchngs_{project_name}",
        experiment_name="nf-core/fetchngs",
        status=final_status,
        parameters={
            "accession_ids": accession_ids[:200],
            "output_dir": output_dir,
            "download_method": download_method,
            "genome": genome,
            "pipeline_version": pipeline_version,
            "nf_core_pipeline": nf_core_pipeline,
        },
        metrics={
            "n_accessions": float(n_accessions),
            "n_samples_downloaded": float(n_samples_downloaded or 0),
            "elapsed_minutes": run_elapsed,
        },
        start_time=run_started_at,
        end_time=run_completed_at,
        tags={
            "project_name": project_name,
            "qc_passed": str(qc_passed),
            "downstream_triggered": str(downstream_triggered),
        },
    )

    # ── Flush to Delta ──
    result = lineage.flush()
    print(f"\n\U0001f4cb Lineage logged: {result['assets_written']} assets, "
          f"{result['edges_written']} edges, {result['run_written']} run")

except ImportError:
    print("\u26a0\ufe0f  genesis_workbench.lineage not available — skipping lineage logging")
    print("   Install wheel v0.1.3+ to enable metadata tier integration")
except Exception as e:
    # Non-fatal: lineage logging should never break the pipeline
    print(f"\u26a0\ufe0f  Lineage logging failed (non-fatal): {e}")

In [0]:
print("\u2554" + "\u2550"*58 + "\u2557")
print("\u2551  Sandwich Wrapper \u2014 fetchngs Run Complete" + " "*13 + "\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Run ID:       {run_id:<42}\u2551")
print(f"\u2551  Pipeline:     nf-core/fetchngs v{pipeline_version:<24}\u2551")
print(f"\u2551  Download:     {download_method:<42}\u2551")
print(f"\u2551  Target:       nf-core/{nf_core_pipeline:<35}\u2551")
print(f"\u2551  Duration:     {run_elapsed:.1f} min (NF: {nf_elapsed/60:.1f} min){' '*(22-len(f'{run_elapsed:.1f} min (NF: {nf_elapsed/60:.1f} min)'))}\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Status:       {final_status.upper():<42}\u2551")
print(f"\u2551  Samples:      {n_samples_downloaded!s:<42}\u2551")
print(f"\u2551  Accessions:   {len(validated_ids)!s:<42}\u2551")
print(f"\u2551  QC gates:     {'PASSED' if qc_passed else 'FAILED':<42}\u2551")
print(f"\u2551  Downstream:   {'Triggered (run ' + str(downstream_run_id) + ')' if downstream_triggered else 'Not triggered':<42}\u2551")
print("\u255a" + "\u2550"*58 + "\u255d")

if qc_reasons:
    print(f"\nQC issues:")
    for r in qc_reasons:
        print(f"  \u2022 {r}")

if samplesheet_path:
    print(f"\nSamplesheet:")
    print(f"  \U0001f4c4 {samplesheet_path}")

print(f"\nArtifacts:")
print(f"  Log:       {log_path}")
print(f"  Config:    {eisai_config_path}")
print(f"  Trace:     {output_dir}/trace/")
print(f"  IDs file:  {ids_file}")

if fastq_catalog:
    total_gb = sum(s["size_mb"] for s in fastq_catalog) / 1024
    print(f"\nDownloaded FASTQs: {total_gb:.2f} GB across {n_samples_downloaded} samples")

# Fail the notebook cell if pipeline failed (so Jobs API sees it)
if nf_exit_code != 0:
    raise RuntimeError(f"nf-core/fetchngs failed with exit code {nf_exit_code}. See log: {log_path}")

---
## Running the Full Pipeline

### Prerequisites

| Requirement | Details |
| --- | --- |
| **Init script** | `install_nextflow.sh` deployed to `/Volumes/dhbl_discovery_us_dev/genesis_schema/scanpy_reference/init_scripts/` |
| **Cluster** | `r5.4xlarge` single-node (16 vCPU, 128 GB RAM, 200 GB GP3 EBS) |
| **Runtime** | DBR `16.4.x-scala2.12` (standard, not GPU/ML) |
| **Security** | `SINGLE_USER` mode (required for UC Volume access) |
| **Network** | Outbound access to SRA/ENA/NCBI (ports 80/443, Aspera port 33001) |

### Option 1: Streamlit UI (Recommended)

1. Navigate to **Single Cell → Data Ingestion → Fetch Public Data (fetchngs)**
2. Paste accession IDs (SRR, SRP, PRJNA, GSE, syn) — one per line or comma-separated
3. Click **Validate & Classify** to verify accessions
4. Click **Fetch ENA Metadata** to preview study size and samples
5. Fill the launch form: output dir, project name, downstream pipeline, download method
6. Click **Launch fetchngs Pipeline** — a job run is submitted automatically

### Option 2: Jobs API (Programmatic)

```python
from utils.fetchngs_pipeline import start_fetchngs_job

job_id, run_id = start_fetchngs_job(
    accession_ids=["PRJNA544731", "SRR9123032"],
    output_dir="/Volumes/dhbl_discovery_us_dev/genesis_schema/scanpy_reference/my_project",
    nf_core_pipeline="scrnaseq",
    download_method="sratools",
    pipeline_version="1.12.0",
    genome="GRCh38",
    trigger_downstream=True,
    project_name="ms_cortex_study",
)
print(f"Run submitted: {run_id}")
```

### Option 3: DABs Job (Bundle Deploy)

```bash
# From bundle root:
cd genesis-workbench/modules/single_cell/scanpy/scanpy_v0.0.1

# Deploy the fetchngs job definition
databricks bundle deploy --target prod

# Run with parameters
databricks bundle run fetchngs --target prod \
  -p accession_ids="SRR9123032" \
  -p output_dir="/Volumes/dhbl_discovery_us_dev/genesis_schema/scanpy_reference/test" \
  -p project_name="test_run" \
  -p nf_core_pipeline="scrnaseq"
```

### Option 4: Direct Notebook Run (Testing)

Attach to a cluster with the Nextflow init script, then fill widgets at the top and **Run All**.

### Cluster Configuration (Job Definition)

```yaml
# resources/fetchngs.job.yml
new_cluster:
  spark_version: 16.4.x-scala2.12
  node_type_id: r5.4xlarge        # 16 vCPU, 128 GB RAM
  num_workers: 0                   # Single-node
  spark_conf:
    spark.master: "local[*]"
    spark.databricks.cluster.profile: singleNode
    spark.driver.memory: 96g
  enable_elastic_disk: true
  aws_attributes:
    ebs_volume_type: GENERAL_PURPOSE_SSD
    ebs_volume_count: 1
    ebs_volume_size: 200           # GB — for FASTQ work dir
  init_scripts:
    - volumes:
        destination: /Volumes/.../init_scripts/install_nextflow.sh
  data_security_mode: SINGLE_USER
```

### End-to-End Flow

```
Accession IDs (SRR, GSE, PRJNA...)
  │
  ▼ [Pre-flight]
  Validate → Generate eisai_fetchngs.config → Register audit
  │
  ▼ [Black Box]
  nextflow run nf-core/fetchngs -r 1.12.0 -c eisai_fetchngs.config
  │
  ▼ [Post-flight]
  Parse trace → Catalog FASTQs → QC gates → Auto-trigger scrnaseq/rnaseq
```

### Delta Tables Written

| Table | Content | Populated At |
| --- | --- | --- |
| `nextflow_run_audit` | Run metadata, status, timing | Pre-flight (insert) + Post-flight (merge) |
| `nextflow_execution_traces` | Per-process resource usage | Post-flight |
| `fetchngs_samples` | Downloaded FASTQ catalog (paths, sizes) | Post-flight |
| `fetchngs_metadata` | Raw SRA/ENA metadata | Post-flight |

### Monitoring a Run

```python
# Check status of a submitted run
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
run = w.jobs.get_run(run_id=<RUN_ID>)
print(f"Status: {run.state.life_cycle_state.value}")
print(f"Result: {run.state.result_state.value if run.state.result_state else 'pending'}")
```

Or query the audit table:
```sql
SELECT run_id, status, n_samples, elapsed_minutes, qc_gate_passed
FROM dhbl_discovery_us_dev.genesis_schema.nextflow_run_audit
WHERE pipeline = 'nf-core/fetchngs'
ORDER BY started_at DESC LIMIT 10;
```

### Supported Accession Types

| Type | Pattern | Repository | Example | Notes |
| --- | --- | --- | --- | --- |
| Run | `SRR/ERR/DRR` + digits | SRA/ENA/DDBJ | SRR9123032 | Single sample |
| Study | `SRP/ERP/DRP` + digits | SRA/ENA/DDBJ | SRP188369 | All runs in study |
| BioProject | `PRJNA/PRJEB/PRJDB` + digits | NCBI/EBI/DDBJ | PRJNA544731 | All runs in project |
| GEO Series | `GSE` + digits | NCBI GEO | GSE204716 | Resolved to SRP |
| Synapse | `syn` + digits | Synapse | syn1234567 | Requires SYNAPSE_AUTH_TOKEN |

### Download Methods Comparison

| Method | Speed | Reliability | Requires |
| --- | --- | --- | --- |
| `sratools` (default) | Fast (parallel) | High (resume, MD5) | SRA Toolkit (in init script) |
| `aspera` | Fastest (50+ MB/s) | Medium (requires license) | Aspera CLI + key |
| `ftp` | Slow (10 MB/s) | High (simple, retryable) | Network access only |

---
## Customization Guide

### Adding Custom Behavior (Zero nf-core Modifications)

All customization flows through `generate_eisai_fetchngs_config()` in the pre-flight layer:

| I want to... | How | Example |
|---|---|---|
| Tune download resources | Add `withName:` block in config | `withName: 'SRATOOLS_FASTERQDUMP' { cpus = 8; memory = '32.GB' }` |
| Increase retry attempts | Override `maxRetries` in selector | `withName: 'ASPERA_CLI' { maxRetries = 5 }` |
| Use a custom container | Override `container` in selector | `withName: 'SRATOOLS_FASTERQDUMP' { container = 'ecr.aws/eisai/sratools:3.1' }` |
| Route outputs to a specific Volume | Use `--outdir` widget | Set `output_dir = "/Volumes/catalog/schema/volume/project"` |
| Skip MD5 validation | Use nf-core `--skip_*` params | `extra_args = "--skip_fastq_check"` (passed as widget) |
| Add a custom QC gate | Edit the post-flight QC cell | Add a new check in the `qc_gate_enabled` block |
| Change the downstream trigger | Edit post-flight trigger cell | Add new wrapper notebook path to `WRAPPER_NOTEBOOKS` dict |

### Version Upgrade Workflow

```
1. Check nf-core changelog:
   https://github.com/nf-core/fetchngs/releases

2. Review for breaking changes:
   - Renamed processes? → Update withName selectors in generate_eisai_fetchngs_config()
   - Renamed params?    → Update the cmd list in Layer 2
   - New processes?     → Optionally add resource selectors
   - Removed processes? → Selectors are harmlessly ignored

3. Bump the version pin:
   - Default widget value in this notebook: pipeline_version = "X.Y.Z"
   - Streamlit form default: same

4. Test with a small sample:
   - Use 2-3 accession IDs from a known study
   - Verify samplesheet is generated
   - Check QC gates pass
   - Verify execution traces in Delta

5. Roll out:
   - Update bundle: databricks bundle deploy --target prod
   - Streamlit app picks up new defaults on next load
```

### Supported Accession Types

| Pattern | Repository | Scope | Example |
|---------|-----------|-------|---------|
| `SRR\d+` / `ERR\d+` / `DRR\d+` | SRA / ENA / DDBJ | Single run (one sample) | SRR12345678 |
| `SRP\d+` / `ERP\d+` / `DRP\d+` | SRA / ENA / DDBJ | Entire study (all runs) | SRP123456 |
| `PRJNA\d+` / `PRJEB\d+` / `PRJDB\d+` | NCBI / EBI / DDBJ | BioProject (all studies) | PRJNA123456 |
| `GSE\d+` | NCBI GEO | GEO Series (all samples) | GSE204716 |
| `syn\d+` | Synapse | Synapse entity | syn1234567 |

### Delta Tables Created/Updated

| Table | Purpose | Updated By |
|---|---|---|
| `genesis_schema.nextflow_run_audit` | Run metadata, status, QC results | Pre-flight (insert) + Post-flight (merge) |
| `genesis_schema.nextflow_execution_traces` | Per-process resource usage from NF trace | Post-flight (append) |
| `genesis_schema.fetchngs_samples` | Downloaded FASTQ catalog with sizes and paths | Post-flight (append) |
| `genesis_schema.fetchngs_metadata` | Raw SRA/ENA metadata for all accessions | Post-flight (append) |

### Download Method Selection

| Method | Best For | Notes |
|--------|----------|-------|
| `sratools` | Default, most reliable | Uses prefetch + fasterq-dump; handles .sra → .fastq conversion |
| `aspera` | Large datasets (>100 GB) | High-speed Aspera protocol; requires ascp client |
| `ftp` | Fallback when others fail | Standard FTP; slowest but most compatible |

These tables enable operational dashboards (download success rates, data volumes, downstream trigger chains) without touching nf-core.